# RAG mit dataset_chunks

Dieses Notebook verwendet die vorhandenen Chunks aus `dataset_chunks` als Wissensbasis für Retrieval-Augmented Generation.

Zuerst werden Embeddings für die Chunks berechnet. Danach werden zu einer Frage die relevantesten Chunks gesucht und als Kontext an ein Sprachmodell übergeben. So kann das Modell Antworten auf Basis deiner Dokumente erzeugen, ohne dass das Modell selbst neu trainiert werden muss.

In [ ]:
from pathlib import Path
from datasets import load_from_disk

DATASET_DIR = Path("dataset_chunks")
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

chunk_ds = load_from_disk(str(DATASET_DIR))
print(chunk_ds)
print(chunk_ds[0])

## Embeddings berechnen

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
texts = chunk_ds["chunk"]
embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

print("Embeddings Shape:", embeddings.shape)

## Retrieval-Funktion

In [ ]:
def search_chunks(question, top_k=3):
    q_emb = embedding_model.encode([question], convert_to_numpy=True)[0]

    doc_norms = np.linalg.norm(embeddings, axis=1)
    q_norm = np.linalg.norm(q_emb)
    scores = (embeddings @ q_emb) / (doc_norms * q_norm + 1e-12)

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "path": chunk_ds[idx].get("path", ""),
            "chunk_id": chunk_ds[idx].get("chunk_id", -1),
            "chunk": chunk_ds[idx]["chunk"],
            "score": float(scores[idx]),
        })
    return results

## Sprachmodell laden

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

print("LLM geladen")

## Prompting mit Kontext

In [ ]:
def build_prompt(question, contexts):
    context_text = "\n\n".join(
        [f"Quelle: {c['path']} | Chunk: {c['chunk_id']}\n{c['chunk']}" for c in contexts]
    )

    prompt = f"""Du beantwortest Fragen nur auf Basis des folgenden Kontexts.
Wenn die Antwort im Kontext nicht enthalten ist, sage klar, dass sie im Kontext nicht gefunden wurde.

Kontext:
{context_text}

Frage:
{question}

Antwort:
"""
    return prompt

In [ ]:
def answer_with_rag(question, top_k=3, max_new_tokens=200):
    contexts = search_chunks(question, top_k=top_k)
    prompt = build_prompt(question, contexts)

    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = generated[len(prompt):].strip() if generated.startswith(prompt) else generated

    return {
        "question": question,
        "contexts": contexts,
        "prompt": prompt,
        "answer": answer,
    }

## Beispielabfragen

In [ ]:
result = answer_with_rag("Was ist LernMAAS?", top_k=3)

print("Frage:", result["question"])
print()
print("Verwendete Kontexte:")
for c in result["contexts"]:
    print(f"- {c['path']} | Chunk {c['chunk_id']} | Score {c['score']:.3f}")
print()
print("Antwort:")
print(result["answer"])

In [ ]:
result = answer_with_rag("Was ist die Aufgabe von LernCloud?", top_k=3)

print("Frage:", result["question"])
print()
print("Verwendete Kontexte:")
for c in result["contexts"]:
    print(f"- {c['path']} | Chunk {c['chunk_id']} | Score {c['score']:.3f}")
print()
print("Antwort:")
print(result["answer"])